# Mini DDPM — Fashion-MNIST (Colab GPU T4)

Notebook ini hanya **menjalankan** kode di `src/`. Semua logika ada di modul Python,
bukan di notebook — supaya tidak ada dua sumber kebenaran.

**Urutan:** clone repo → cek GPU → mount Drive → training → sampling → unduh hasil.

> Runtime → Change runtime type → **T4 GPU** sebelum mulai.

In [ ]:
# 1. Clone repo (Private: butuh PAT, atau pakai 'Save a copy in GitHub' dari Colab)
REPO_URL = "https://github.com/USERNAME/miniddpm-fashionmnist.git"

import os
if not os.path.isdir("MiniDDPM-FashionMNIST"):
    !git clone {REPO_URL} MiniDDPM-FashionMNIST
%cd MiniDDPM-FashionMNIST
!git log --oneline -5

In [ ]:
# 2. Cek GPU + pasang dependensi
!nvidia-smi
!pip install -q torch torchvision tqdm matplotlib

import torch
print("torch :", torch.__version__)
print("CUDA  :", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU   :", torch.cuda.get_device_name(0))

In [ ]:
# 3. Mount Drive untuk checkpoint (sesi Colab bisa putus kapan saja)
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/miniddpm'
os.makedirs(DRIVE_DIR, exist_ok=True)
print("Checkpoint akan disalin ke:", DRIVE_DIR)

In [ ]:
# 4. Konfigurasi
from src.config import Config

cfg = Config()
cfg.epochs = 40          # T4 jauh lebih cepat dari CPU: 25 → 40 epoch
cfg.unet_base = 32
cfg.batch_size = 128
cfg.subset_size = 15000  # naikkan ke 60000 kalau waktu masih cukup

print(cfg.run_name())
cfg.to_dict()

In [ ]:
# 5. Training
from src.train import train

train(cfg)

In [ ]:
# 6. Salin checkpoint ke Drive (jalankan setelah training selesai)
!mkdir -p {DRIVE_DIR}/checkpoints {DRIVE_DIR}/logs {DRIVE_DIR}/figures
!cp -v checkpoints/*.pt {DRIVE_DIR}/checkpoints/ 2>/dev/null | tail -3
!cp -v logs/*.json {DRIVE_DIR}/logs/ 2>/dev/null | tail -3
print("Selesai. Isi Drive:")
!du -sh {DRIVE_DIR}/*

In [ ]:
# 7. Sampling + grid progres antar-epoch
from src.sample import build_epoch_progression_grid

# target_epochs=None = semua checkpoint; isi list untuk membatasi tinggi grid
build_epoch_progression_grid(cfg, target_epochs=None, num_samples=8, fixed_seed=1234)

In [ ]:
# 8. Tampilkan grid di notebook + salin ke Drive
import glob
from IPython.display import Image, display

for p in sorted(glob.glob('report/figures/*.png'))[-3:]:
    print(p)
    display(Image(p))

!cp -v report/figures/*.png {DRIVE_DIR}/figures/ 2>/dev/null | tail -3

In [ ]:
# 9. Kurva loss (dari logs/train_history.json)
import json, matplotlib.pyplot as plt

hist = json.load(open('logs/train_history.json'))
# train.py menyimpan list of dict: [{'epoch':..,'loss':..,'time_sec':..}, ...]
losses = [h['loss'] for h in hist]

plt.figure(figsize=(7, 3.5))
plt.plot(range(1, len(losses) + 1), losses, marker='o', ms=3)
plt.xlabel('Epoch'); plt.ylabel('MSE loss'); plt.grid(alpha=.3)
plt.title('Kurva training DDPM')
plt.tight_layout()
plt.savefig('report/figures/loss_curve.png', dpi=150)
plt.show()

## Catatan

- Kalau sesi putus di tengah training, jalankan ulang sel 1–4, lalu salin balik checkpoint dari Drive ke `checkpoints/` sebelum lanjut.
- `num_workers=0` dipertahankan meski di Linux boleh >0 — dataset 21 MB sudah di cache, dan menghindari bug fork membuat hasil reproducible.
- Jangan commit notebook ini bersamaan dengan orang lain (format `.ipynb` = JSON, rawan konflik). Satu orang pegang notebook.